# Fitting QENS data with EasyDynamics

Previously, some quasi-elastic neutron scattering (QENS) data has been [simulated](./../3-mcstas/mcstas-qens.ipynb) and [reduced](./../4-reduction/reduction-qens.ipynb), and can now be analysed with [`easydynamics`](https://easyscience.github.io/dynamics-lib/).

In the reduction step we produced two data sets:

- an **elastic** sample, which scatters neutrons without changing their energy. We will use it to measure the **instrument resolution**.
- a **quasi-elastic** sample, in which the scatterers diffuse. This is the data we actually want to interpret.

The workflow is as follows: We first load the data into EasyDynamics and inspect it. We then determine the resolution function using the elastic data. Next, we fit the QENS data to an empirical model and inspect the resulting fit. Finally, we extract physically meaningful information from these fits.

In [ ]:
import os

import numpy as np
import scipp as sc

import easydynamics as edyn
from easydynamics.experiment import Experiment
import easydynamics.sample_model as sm

# Quiz questions for this notebook
from jupyterquiz import display_quiz
from quizlib import qens as quiz

# Make the plots interactive
%matplotlib widget

## Load and prepare the data

First we load the **elastic** sample data. We will use it to determine the instrument resolution, in the same way that a vanadium measurement is used in a real experiment.

ℹ️ **If you did not complete the QENS data reduction yesterday**, don't worry: the loading cells below automatically download pre-prepared data when the reduced files are not found.

Our reduced data are histograms of neutron counts, and they need a small adjustment before fitting. Future versions of EasyDynamics will not need this step.

Some of the bins have zero counts, and consequently zero variance. When fitting, each data point is weighted by 1/variance, which therefore diverges, giving those bins infinite weight and destabilising the fit. The solution that we will employ is to give all these bins a variance of 1 instead.

In [ ]:
def prepare_data(experiment):
    """Give bins with too small variance a variance of 1, so that their weight in the fit stays finite."""
    data = experiment.data
    indices = data.variances <= 0.5
    data.variances[indices] = 1.0
    experiment.data = data


In [ ]:
display_quiz(quiz.q1)

In [ ]:
filename = '../4-reduction/energy_QENS_elastic.h5'
if not os.path.exists(filename):
    # Fall back to pre-prepared data if the reduction exercise was not completed
    import utils
    filename = utils.fetch_data('4-reduction/energy_QENS_elastic.h5')

elastic_experiment = Experiment('Elastic')
elastic_experiment.load_hdf5(filename=filename)
prepare_data(elastic_experiment)

elastic_experiment.plot_data(slicer=True)

Look closely at the position of the elastic peak in the plot above (you may need to zoom in).

In [ ]:
display_quiz(quiz.q2)

Next we load the **quasi-elastic** sample data. This is the data whose dynamics we want to understand.

In [ ]:
filename2 = '../4-reduction/energy_QENS_sample.h5'
if not os.path.exists(filename2):
    # Fall back to pre-prepared data if the reduction exercise was not completed
    import utils
    filename2 = utils.fetch_data('4-reduction/energy_QENS_sample.h5')

qe_experiment = Experiment('QuasiElastic')
qe_experiment.load_hdf5(filename=filename2)
prepare_data(qe_experiment)

qe_experiment.plot_data(slicer=True)

## Step 1: Determine the resolution from the elastic sample

The scattering from the elastic sample is, to a very good approximation, a delta function in energy transfer: the neutrons come out with the same energy they went in with. The width we actually measure is therefore entirely due to the **instrument resolution**.

In [ ]:
display_quiz(quiz.q3)

We model the resolution as a single `Gaussian`. We collect it in a `ComponentCollection` and use that to build a `SampleModel`. (In a real experiment the resolution may need several Gaussians or other shapes to be described accurately.)
Note that we do not give the Gaussian a center. This is because it describes elastic scattering, and the offset we see in the data is handled by the InstrumentModel, introduced below.

In [ ]:
resolution_components = sm.ComponentCollection(y_unit='counts')
res_gauss = sm.Gaussian(width=0.002, area=1, name='Res. Gauss', y_unit='counts')
resolution_components.append_component(res_gauss)

resolution_sample_model = sm.SampleModel(components=resolution_components, y_unit='counts')

In [ ]:
display_quiz(quiz.q4)

Although the background in this simulated data is essentially zero, we still show how to add a `BackgroundModel`, as we would for real data. We use a `Polynomial` with a single coefficient, i.e. a flat background. Because the background is zero, we fix this parameter so it cannot vary — one should never give a model free parameters that are not needed.

In [ ]:
poly_res = sm.Polynomial(coefficients=[0.0], name='Background', y_unit='counts')
poly_res.coefficients[0].min = 0.0
poly_res.coefficients[0].fixed = True
background_model_res = sm.BackgroundModel(components=poly_res, y_unit='counts')

The background model goes into an `InstrumentModel`. This model also contains a fittable energy offset that accounts for the misalignment of the instrument that we found above; all components are centred on this offset.

In [ ]:
instrument_model_res = sm.InstrumentModel(
    energy_offset=1e-3,
    x_unit='meV',
    background_model=background_model_res,
)

We now collect everything in an `Analysis` object: a display name, the experiment, the sample model and the instrument model. EasyDynamics automatically generates a model for each `Q` value in the data.

In [ ]:
elastic_analysis = edyn.Analysis(
    display_name='Elastic / Resolution',
    experiment=elastic_experiment,
    sample_model=resolution_sample_model,
    instrument_model=instrument_model_res,
)

Let us first fit a single `Q` index and plot the data and model to see how it looks. We use the `independent` fit method for one arbitrary `Q` index.

In [ ]:

elastic_analysis.fit(fit_method='independent', Q_index=2)
elastic_analysis.plot_data_and_model(Q_index=2)

The fit looks good, so let us fit all `Q` indices independently and plot the results.

In [ ]:
elastic_analysis.fit(fit_method='independent')
elastic_analysis.plot_data_and_model()

It is useful to inspect the fitted parameters. We can turn them into a scipp dataset, and we can plot any of them as a function of `Q` with `plot_parameters`. A good resolution function should have a width and area that vary only slowly with `Q`.

In [ ]:
elastic_pars = elastic_analysis.parameters_to_dataset()
elastic_pars

In [ ]:
elastic_analysis.plot_parameters(names=['Res. Gauss width'])

In [ ]:
elastic_analysis.plot_parameters(names=['Res. Gauss area'])

### Normalise to the resolution area

Notice that the area of the resolution is not constant. Since the sample scatters the same in all directions, this variation reflects the instrument (detector size and efficiency, analyzer coverage, etc.) rather than the sample.

How can we deal with this? (There may be multiple ways.)

In [ ]:
display_quiz(quiz.q5)

We will normalise the quasi-elastic data to the resolution area, so we divide it out, just as one would normalise to a vanadium measurement in a real experiment. `sc.values(...)` drops the uncertainties of the fitted areas, so that they act as a fixed normalisation.

⚠️ Run the next cell only once. Running it a second time would divide the data again.

In [ ]:
norm = elastic_pars['Res. Gauss area'].copy()
norm.unit = 'dimensionless'  # to keep the units of the data consistent
qe_experiment.data = qe_experiment.data / sc.values(norm)

Now that we know the instrument resolution, look again at the quasi-elastic data we plotted earlier (note that the intensity scale has changed after the normalisation). It shows a sharp peak and a wider peak. Compare their widths to the width of the resolution.

In [ ]:
qe_experiment.plot_data(slicer=True)

In [ ]:
display_quiz(quiz.q6)

In [ ]:
display_quiz(quiz.q7)

## Step 2: Fit the quasi-elastic sample at each Q

We are now happy with the resolution and can turn to the quasi-elastic sample. Looking at the data we plotted earlier, it has a sharp elastic peak, a broader quasi-elastic peak, and a small flat background.

We describe it with a `SampleModel` containing:

- a `DeltaFunction` for the elastic (immobile) scattering, and
- a `Lorentzian` for the quasi-elastic broadening caused by motion.

### Exercise: create a new SampleModel for the quasi-elastic sample
Follow the steps to create a `SampleModel` like we did above, but this time create both a `DeltaFunction` and a `Lorentzian` and append them. Give the `DeltaFunction` a reasonable start value for the area, and the `Lorentzian` both an area and width (half width at half max). Remember to set the `y_unit` to 'counts' everywhere.

**Solution:**

In [ ]:
sample_components = sm.ComponentCollection(y_unit='counts')
delta_function = sm.DeltaFunction(name='DeltaFunction', area=0.15, y_unit='counts')
lorentzian = sm.Lorentzian(name='Lorentzian', area=1.0, width=0.015, y_unit='counts')

sample_components.append_component(delta_function)
sample_components.append_component(lorentzian)

sample_model_qe = sm.SampleModel(components=sample_components, y_unit='counts')

We build a new `InstrumentModel`, and this time we give it a resolution: the `SampleModel` from our elastic fit. All of its parameters are automatically fixed and the resolution is normalised to have area 1.

In [ ]:

poly_qe = sm.Polynomial(coefficients=[0.0], name='Background', y_unit='counts')
poly_qe.coefficients[0].min = 0.0
poly_qe.coefficients[0].fixed = True
background_model_qe = sm.BackgroundModel(components=poly_qe, y_unit='counts')

instrument_model_qe = sm.InstrumentModel(
    background_model=background_model_qe,
    resolution_model=elastic_analysis.sample_model,
    energy_offset=1e-3,
    x_unit='meV',
)


### Exercise: collect the data, SampleModel and InstrumentModel in an Analysis object

**Solution:**

In [ ]:

qe_analysis = edyn.Analysis(
    display_name='Quasi-elastic per-Q',
    experiment=qe_experiment,
    sample_model=sample_model_qe,
    instrument_model=instrument_model_qe,
)

`Analysis` handles the convolution of the `sample_model` with the resolution. The calculation is analytical where possible and numerical otherwise. 

Before fitting, it is a good idea to check the start guesses by plotting the data together with the model.

In [ ]:
qe_analysis.plot_data_and_model()

Adjust the start guesses until they look reasonable, then fit every `Q` independently and plot the result like we did above.

**Solution:**

In [ ]:
qe_analysis.fit(fit_method='independent')
qe_analysis.plot_data_and_model()

The interesting parameters are the width and area of the `Lorentzian`. Let us plot them as a function of `Q`.

In [ ]:
qe_analysis.plot_parameters(names=['Lorentzian width'], vmin=0, vmax=0.03, xmin=0, xmax=2.1)

In [ ]:
qe_analysis.plot_parameters(names=['Lorentzian area'], vmin=0, vmax=2.0, xmin=0, xmax=2.1)

Sometimes, the fitter produces no error bars on the fit parameters. We are working on fixing this, but have to work around it in the meantime. Run the cell below if needed.

In [ ]:
def fix_missing_variances(analysis):
    """Fix missing variances on fitted parameters in an analysis."""
    for param in analysis.get_all_parameters():
        if param.variance == 0.0:
            param.variance = 0.00001 * abs(param.value)

fix_missing_variances(qe_analysis)


Before reading on, study the plot of the Lorentzian width as a function of `Q`.

In [ ]:
display_quiz(quiz.q8)

In [ ]:
display_quiz(quiz.q9)

## Step 3: Fit a jump-diffusion model to the widths

The width does not simply grow like $Q^2$: it levels off at high $Q$. This is the signature of **jump diffusion**, in which a particle sits still for a residence time $\tau$ and then jumps to a new site. The half-width of the quasi-elastic Lorentzian is

$$
\Gamma(Q) = \frac{\hbar\,D\,Q^2}{1 + D\,\tau\,Q^2},
$$

where $D$ is the diffusion coefficient and $\tau$ is the residence (relaxation) time. At low $Q$ this reduces to ordinary diffusion, $\Gamma \approx \hbar D Q^2$, while at high $Q$ it saturates at the plateau $\hbar/\tau$.

Our width curve shows exactly this behaviour: a $Q^2$ rise that bends over towards a plateau within the measured range, so the data constrain **both** parameters: the low-$Q$ slope fixes $D$, and the high-$Q$ plateau fixes $\tau$.

As a first step we fit the jump-diffusion model to the fitted Lorentzian parameters we obtained above. We create a `JumpTranslationalDiffusion` model whose `lorentzian_name` matches our fitted `Lorentzian`, wrap it in a `FitBinding`, and pass it to a `ParameterAnalysis` together with the fitted parameters from the per-`Q` analysis. Because the model's `lorentzian_name` matches the component, the binding fits both the `Lorentzian width` (giving $D$ and $\tau$) and the `Lorentzian area` (giving the scale).

In [ ]:
jump_diffusion_model = sm.JumpTranslationalDiffusion(
    name='Jump Translational Diffusion',
    lorentzian_name='Lorentzian',
    diffusion_coefficient=4.6e-10,
    relaxation_time=22.0,  # ps
    scale=0.5,
)

binding = edyn.FitBinding(model=jump_diffusion_model)

# Here we need to change the units of the Lorentzian area parameter to meV, so that the units of the diffusion coefficient are correct.
# This is a known issue in the current version of easydynamics, and will be fixed in a future release.
pars = qe_analysis.parameters_to_dataset()
pars['Lorentzian area'].unit = 'meV'

parameter_analysis = edyn.ParameterAnalysis(
    parameters=pars,
    bindings=binding,
)

We first plot the start guess to see if it is reasonable.

In [ ]:
parameter_analysis.plot(names=['Lorentzian width'], xmin=0, xmax=2.1, vmin=0, vmax=0.02)

Then we fit and plot again.

In [ ]:
parameter_analysis.fit()
parameter_analysis.plot(names=['Lorentzian width'], xmin=0, xmax=2.1, vmin=0, vmax=0.02)

We can read off the fitted jump-diffusion parameters, with uncertainties.

In [ ]:
parameter_analysis.get_all_parameters()

## Step 4: Fit the jump-diffusion model to all the data at once

The two-step approach above works, but we can do better. Now that we know the quasi-elastic scattering follows a jump-diffusion model, we can fit that model **directly to the data**, using all `Q` values simultaneously. This uses every data point at once and generally gives smaller uncertainties. In addition to the diffusion, we still describe the elastic incoherent scattering with a `DeltaFunction`.

We build a new `SampleModel` that has a `DeltaFunction` component and a `JumpTranslationalDiffusion` diffusion model, and new `BackgroundModel` and `InstrumentModel` objects.

In [ ]:
delta_function_diff = sm.DeltaFunction(name='DeltaFunction', area=0.15, y_unit='counts')
diffusion_components = sm.ComponentCollection(components=[delta_function_diff], y_unit='counts')

diffusion_model = sm.JumpTranslationalDiffusion(
    name='Jump Translational Diffusion',
    diffusion_coefficient=2.5e-10,
    relaxation_time=18.0,  # ps
    scale=1.0,
    x_unit='meV',
    y_unit='counts',
)

sample_model_diff = sm.SampleModel(
    components=diffusion_components,
    diffusion_models=diffusion_model,
    y_unit='counts',
)


### Exercise: Complete the creation of the analysis object for the Jump Diffusion
Follow the steps above: define an InstrumentModel, where you pass the elastic analysis as the resolution model, consider adding a background, etc.

**Solution:**

In [ ]:

poly_diff = sm.Polynomial(coefficients=[0.0], name='Background', y_unit='counts')
poly_diff.coefficients[0].min = 0.0
poly_diff.coefficients[0].fixed = True
background_model_diff = sm.BackgroundModel(components=poly_diff, y_unit='counts')

instrument_model_diff = sm.InstrumentModel(
    background_model=background_model_diff,
    resolution_model=elastic_analysis.sample_model,
    energy_offset=1e-3,
    x_unit='meV',
)

diffusion_analysis = edyn.Analysis(
    display_name='Jump Diffusion Full Analysis',
    experiment=qe_experiment,
    sample_model=sample_model_diff,
    instrument_model=instrument_model_diff,
)

As always, we check the start guess before fitting.

In [ ]:
diffusion_analysis.plot_data_and_model()

Now we fit all the data simultaneously.

In [ ]:
diffusion_analysis.fit(fit_method='simultaneous')


In [ ]:
diffusion_analysis.plot_data_and_model(plot_residuals=True, autoscale=False)

The diffusion parameters are just a couple of numbers with uncertainties, so instead of plotting them we display them directly. Both the diffusion coefficient $D$ and the residence time $\tau$ are now determined by the data.

In [ ]:
diffusion_model.get_global_variables()

For reference, here are the parameters from the two-step fit (fitting the Lorentzian widths and areas). Notice that fitting all the data simultaneously generally gives smaller uncertainties than the two-step route.

In [ ]:
parameter_analysis.get_all_parameters()

In [ ]:
display_quiz(quiz.q10)

Finally, since we know what we put into McStas we can compare our answers to the true values: $D = 4.6\times10^{-10}$ m$^2$/s and $\tau = 22$ ps. Did you get similar values? Why/why not?

As a further check, the fitted areas have a physical meaning after the normalisation: the `Lorentzian` area (≈ 0.85) and the `DeltaFunction` area (≈ 0.07) correspond to a quasi-elastic fraction of about 92% — matching the `f_QE = 0.92` that was set for the McStas sample.

In [ ]:
display_quiz(quiz.q11)

This is the end of this analysis. EasyDynamics is work in progress, and we would very much like to hear your feedback. Please write to [henrik.jacobsen@ess.eu](mailto:henrik.jacobsen@ess.eu).